# WiSARv1 Dataset Organization Investigation

This notebook performs a **read-only structural investigation** of a WiSARv1 dataset. It inventories paths, directory names, filenames, and metadata/manifests without opening image pixel data, modifying the dataset, copying files, extracting archives, resizing images, or training models.

The outputs are small aggregate text/CSV reports intended to answer whether defensible flight, recording-session, sequence, scene, or collection identifiers exist for leakage-safe splitting. Findings are labeled `documented`, `observed`, or `unknown`; different names are never treated as proof of different flights.

In [17]:
from __future__ import annotations

import csv
import os
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

# Set WISAR_DATASET_ROOT to the mounted dataset directory in Colab.
# Expected ZIP location: /content/drive/MyDrive/WiSARD/WiSARDv1.zip
DATASET_ROOT = Path(os.environ.get("WISAR_DATASET_ROOT", "/content/drive/MyDrive/WiSARD"))
if not DATASET_ROOT.exists() and Path("data/raw/WiSARD").is_dir():
    DATASET_ROOT = Path("data/raw/WiSARD")
REPORT_DIR = Path(os.environ.get("WISAR_REPORT_DIR", "results/dataset_audit"))
ZIP_NAME = "WiSARDv1.zip"
TREE_MAX_DEPTH = 4
MAX_METADATA_BYTES = 2_000_000
MAX_TEXT_LINES_PER_FILE = 2_000

METADATA_EXTENSIONS = {
    ".csv", ".json", ".xml", ".yaml", ".yml", ".txt", ".tsv", ".mat",
    ".ini", ".cfg", ".conf", ".toml", ".md", ".md5", ".log",
}
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
    ".gif", ".ppm", ".pgm", ".dng", ".heic",
}
SEARCH_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "timestamp", "gps", "trajectory", "video", "mission",
)
GROUPING_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "mission", "run", "take", "trip", "set",
)


In [18]:
def tokenize_name(value: str) -> list[str]:
    """Split names into stable lowercase alphanumeric tokens without opening files."""
    return [token for token in re.split(r"[^a-zA-Z0-9]+", value.lower()) if token]


In [19]:
if not DATASET_ROOT.exists() or not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Set DATASET_ROOT to the mounted WiSARv1 directory; not found: {DATASET_ROOT}"
    )

root_resolved = DATASET_ROOT.resolve()
file_records = []
directory_records = []
metadata_records = []
extension_counts = Counter()
extension_by_directory = Counter()
directory_file_counts = Counter()
name_tokens = Counter()
directory_name_tokens = Counter()
tree_lines = [f"{root_resolved.name}/"]

# os.scandir reads directory entries and stat information; it does not decode image pixels.
stack = [(root_resolved, 0)]
while stack:
    current, depth = stack.pop()
    try:
        entries = sorted(os.scandir(current), key=lambda entry: (not entry.is_dir(follow_symlinks=False), entry.name.lower()))
    except (OSError, PermissionError) as error:
        directory_records.append({"relative_directory": str(current.relative_to(root_resolved)), "status": f"unreadable: {error}"})
        continue

    relative_current = current.relative_to(root_resolved)
    directory_records.append({"relative_directory": "." if relative_current == Path(".") else str(relative_current), "status": "read"})
    if depth <= TREE_MAX_DEPTH:
        tree_lines.extend([f"{'  ' * (depth + 1)}{'[D] ' if entry.is_dir(follow_symlinks=False) else '[F] '}{entry.name}" for entry in entries])

    for entry in entries:
        entry_path = Path(entry.path)
        relative_path = entry_path.relative_to(root_resolved)
        if entry.is_dir(follow_symlinks=False):
            directory_name_tokens.update(tokenize_name(entry.name))
            stack.append((entry_path, depth + 1))
            continue
        if not entry.is_file(follow_symlinks=False):
            continue

        suffix = entry_path.suffix.lower() or "[no_extension]"
        relative_directory = str(relative_path.parent)
        extension_counts[suffix] += 1
        extension_by_directory[(relative_directory, suffix)] += 1
        directory_file_counts[relative_directory] += 1
        tokens = tokenize_name(entry.name)
        name_tokens.update(tokens)
        record = {
            "relative_path": str(relative_path),
            "relative_directory": relative_directory,
            "filename": entry.name,
            "extension": suffix,
            "size_bytes": entry.stat(follow_symlinks=False).st_size,
            "name_tokens": ",".join(tokens),
        }
        file_records.append(record)
        if suffix in METADATA_EXTENSIONS:
            metadata_records.append(record.copy())

print(f"Dataset root: {root_resolved}")
print(f"Directories observed: {len(directory_records):,}")
print(f"Files observed: {len(file_records):,}")
print(f"Metadata-like files: {len(metadata_records):,}")
print("No image file was opened as pixel data.")

Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Directories observed: 1
Files observed: 1
Metadata-like files: 0
No image file was opened as pixel data.


In [25]:
zip_path = DATASET_ROOT / ZIP_NAME if DATASET_ROOT.is_dir() else None
zip_member_records = []
zip_metadata_matches = []
zip_metadata_snippets = []
zip_candidate_rows = []
zip_sensor_rows = []
zip_pattern_counts = Counter()
zip_pattern_examples = defaultdict(list)
zip_extension_counts = Counter()
zip_tree_lines = []
zip_status = "unknown"

if zip_path is not None and zip_path.is_file():
    zip_status = "observed"
    # ZipFile reads the archive directory and selected metadata members only; it never extracts files.
    with zipfile.ZipFile(zip_path, mode="r") as archive:
        infos = archive.infolist()
        member_pairs = [
            (info, info.filename.replace("\\", "/").strip("/"))
            for info in infos
            if info.filename.strip("/")
        ]
        zip_member_records = []
        for info, member_path in member_pairs:
            if not member_path:
                continue
            suffix = Path(member_path).suffix.lower() or "[no_extension]"
            is_directory = info.is_dir() or info.filename.endswith(("/", "\\"))
            if not is_directory:
                zip_extension_counts[suffix] += 1
            zip_member_records.append({
                "member_path": member_path,
                "relative_directory": str(Path(member_path).parent),
                "filename": Path(member_path).name,
                "extension": suffix,
                "size_bytes": info.file_size,
                "compressed_size_bytes": info.compress_size,
                "is_directory": is_directory,
            })

        tree_nodes = {"": {"directories": set(), "files": set()}}
        for record in zip_member_records:
            parts = Path(record["member_path"]).parts
            for index in range(len(parts)):
                parent = "/".join(parts[:index])
                node = parts[index]
                tree_nodes.setdefault(parent, {"directories": set(), "files": set()})
                if index < len(parts) - 1 or record["is_directory"]:
                    tree_nodes.setdefault(parent, {"directories": set(), "files": set()})["directories"].add(node)
                else:
                    tree_nodes.setdefault(parent, {"directories": set(), "files": set()})["files"].add(node)

        zip_tree_lines = [f"{DATASET_ROOT.name}/{ZIP_NAME}"]
        for parent, node in sorted(tree_nodes.items()):
            depth = 0 if not parent else len(Path(parent).parts)
            if depth > TREE_MAX_DEPTH:
                continue
            prefix = "  " * (depth + 1)
            for directory in sorted(node["directories"]):
                zip_tree_lines.append(f"{prefix}[D] {directory}")
            for filename in sorted(node["files"]):
                zip_tree_lines.append(f"{prefix}[F] {filename}")
else:
    zip_status = "unknown"

print(f"ZIP path: {zip_path}")
print(f"ZIP status: {zip_status}")
print(f"ZIP members observed: {len(zip_member_records):,}")
print("ZIP extracted: no; image pixels opened: no")

ZIP path: d:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD\WiSARDv1.zip
ZIP status: unknown
ZIP members observed: 0
ZIP extracted: no; image pixels opened: no


In [13]:
metadata_matches = []
metadata_snippets = []
pattern_counts = Counter()
pattern_examples = defaultdict(list)

for record in metadata_records:
    path = root_resolved / record["relative_path"]
    filename_lower = record["filename"].lower()
    filename_terms = [term for term in SEARCH_TERMS if term in filename_lower]
    content_terms = []
    content = ""
    content_status = "not_read"
    if record["extension"] != ".mat" and record["size_bytes"] <= MAX_METADATA_BYTES:
        try:
            content = path.read_text(encoding="utf-8", errors="replace")
            content_status = "read"
        except (OSError, UnicodeError) as error:
            content_status = f"unreadable: {error}"
    elif record["extension"] == ".mat":
        content_status = "binary_mat_not_decoded"
    else:
        content_status = "skipped_over_size_limit"

    if content:
        content_lower = content.lower()
        content_terms = [term for term in SEARCH_TERMS if term in content_lower]
        for line_number, line in enumerate(content.splitlines()[:MAX_TEXT_LINES_PER_FILE], start=1):
            line_terms = [term for term in SEARCH_TERMS if term in line.lower()]
            if line_terms and len(metadata_snippets) < 200:
                metadata_snippets.append({
                    "relative_path": record["relative_path"],
                    "line_number": line_number,
                    "terms": ",".join(line_terms),
                    "snippet": re.sub(r"\s+", " ", line.strip())[:240],
                })

    all_terms = sorted(set(filename_terms + content_terms))
    if all_terms:
        evidence_status = "documented" if content_terms else "observed"
        metadata_matches.append({
            "relative_path": record["relative_path"],
            "extension": record["extension"],
            "filename_terms": ",".join(filename_terms),
            "content_terms": ",".join(content_terms),
            "terms": ",".join(all_terms),
            "content_status": content_status,
            "evidence_status": evidence_status,
        })

for record in file_records:
    source_name = f"{record['relative_directory']}/{record['filename']}"
    for pattern, label in (
        (r"(?:^|[^a-zA-Z])(?:flight|sequence|session|recording|collection|scene|mission|run|take|trip|set)[-_]?[a-zA-Z0-9]+", "grouping_term_with_value"),
        (r"(?:^|[^a-zA-Z])\d{2,}(?:[^a-zA-Z]|$)", "numeric_identifier"),
        (r"[A-Za-z]+[_-]\d+", "label_number"),
    ):
        matches = re.findall(pattern, source_name, flags=re.IGNORECASE)
        if matches:
            pattern_counts[label] += len(matches)
            for match in matches[:3]:
                if len(pattern_examples[label]) < 10:
                    pattern_examples[label].append(match.strip(" _-"))

candidate_rows = []
for record in file_records:
    path_parts = Path(record["relative_path"]).parts
    for part in path_parts:
        part_lower = part.lower()
        matched_terms = [term for term in GROUPING_TERMS if term in part_lower]
        has_identifier_shape = bool(re.search(r"(?:^|[_-])(?:\d+|[a-z]+\d+)(?:$|[_-])", part_lower))
        if matched_terms or has_identifier_shape:
            candidate_rows.append({
                "candidate": part,
                "source_path": record["relative_path"],
                "matched_terms": ",".join(matched_terms),
                "evidence_status": "observed",
                "interpretation": "Naming/path pattern only; not proof of an independent flight or session.",
            })

# Canonicalize RGB/thermal paths only for comparison; this does not alter source paths.
sensor_groups = defaultdict(lambda: {"rgb": set(), "thermal": set()})
for record in file_records:
    parts = list(Path(record["relative_path"]).parts)
    sensors = {part.lower() for part in parts if part.lower() in {"rgb", "thermal", "visible", "infrared", "ir"}}
    if not sensors:
        continue
    sensor = "rgb" if "rgb" in sensors or "visible" in sensors else "thermal"
    canonical_parts = [part.lower() for part in parts if part.lower() not in {"rgb", "thermal", "visible", "infrared", "ir"}]
    canonical = "/".join(canonical_parts[:-1]) if canonical_parts else "."
    sensor_groups[canonical][sensor].add(record["filename"].lower())

sensor_rows = []
for canonical, groups in sorted(sensor_groups.items()):
    has_both = bool(groups["rgb"] and groups["thermal"])
    sensor_rows.append({
        "canonical_path_without_sensor": canonical,
        "rgb_file_count": len(groups["rgb"]),
        "thermal_file_count": len(groups["thermal"]),
        "shared_filename_count": len(groups["rgb"] & groups["thermal"]),
        "evidence_status": "observed" if has_both else "unknown",
        "interpretation": (
            "RGB and thermal occur under a shared canonical path; verify timestamps/metadata before grouping."
            if has_both else
            "No paired path observed here; this does not prove different flights or sessions."
        ),
    })

print(f"Metadata files matching investigation terms: {len(metadata_matches):,}")
print(f"Candidate grouping path/name observations: {len(candidate_rows):,}")
print(f"RGB/thermal canonical groups: {len(sensor_rows):,}")

Metadata files matching investigation terms: 0
Candidate grouping path/name observations: 0
RGB/thermal canonical groups: 0


In [22]:
if zip_status == "observed":
    with zipfile.ZipFile(zip_path, mode="r") as archive:
        info_by_path = {info.filename.replace("\\", "/").strip("/"): info for info in archive.infolist()}
        for record in zip_member_records:
            member_path = record["member_path"]
            path_lower = member_path.lower()
            filename_terms = [term for term in SEARCH_TERMS if term in path_lower]
            content_terms = []
            content_status = "not_read"
            content = ""
            info = info_by_path.get(member_path)
            is_safe_metadata = record["extension"] in METADATA_EXTENSIONS and record["extension"] not in IMAGE_EXTENSIONS
            if info and not record["is_directory"] and is_safe_metadata and info.file_size <= MAX_METADATA_BYTES:
                try:
                    with archive.open(info, mode="r") as metadata_handle:
                        content = metadata_handle.read(MAX_METADATA_BYTES).decode("utf-8", errors="replace")
                    content_status = "read_from_zip_without_extraction"
                except (OSError, RuntimeError, UnicodeError) as error:
                    content_status = f"unreadable: {error}"
            elif record["extension"] == ".mat":
                content_status = "binary_mat_not_decoded"
            elif record["is_directory"]:
                content_status = "directory_member"
            elif record["extension"] in METADATA_EXTENSIONS:
                content_status = "skipped_over_size_limit"

            if content:
                content_terms = [term for term in SEARCH_TERMS if term in content.lower()]
                for line_number, line in enumerate(content.splitlines()[:MAX_TEXT_LINES_PER_FILE], start=1):
                    line_terms = [term for term in SEARCH_TERMS if term in line.lower()]
                    if line_terms and len(zip_metadata_snippets) < 200:
                        zip_metadata_snippets.append({
                            "member_path": member_path,
                            "line_number": line_number,
                            "terms": ",".join(line_terms),
                            "snippet": re.sub(r"\s+", " ", line.strip())[:240],
                        })

            all_terms = sorted(set(filename_terms + content_terms))
            if all_terms:
                zip_metadata_matches.append({
                    "member_path": member_path,
                    "extension": record["extension"],
                    "filename_terms": ",".join(filename_terms),
                    "content_terms": ",".join(content_terms),
                    "terms": ",".join(all_terms),
                    "content_status": content_status,
                    "evidence_status": "documented" if content_terms else "observed",
                })

            for pattern, label in (
                (r"(?:^|[^a-zA-Z])(?:flight|sequence|session|recording|collection|scene|mission|run|take|trip|set)[-_]?[a-zA-Z0-9]+", "grouping_term_with_value"),
                (r"(?:^|[^a-zA-Z])\d{2,}(?:[^a-zA-Z]|$)", "numeric_identifier"),
                (r"[A-Za-z]+[_-]\d+", "label_number"),
            ):
                matches = re.findall(pattern, member_path, flags=re.IGNORECASE)
                if matches:
                    zip_pattern_counts[label] += len(matches)
                    for match in matches[:3]:
                        if len(zip_pattern_examples[label]) < 10:
                            zip_pattern_examples[label].append(match.strip(" _-"))

            for part in Path(member_path).parts:
                part_lower = part.lower()
                matched_terms = [term for term in GROUPING_TERMS if term in part_lower]
                has_identifier_shape = bool(re.search(r"(?:^|[_-])(?:\d+|[a-z]+\d+)(?:$|[_-])", part_lower))
                if matched_terms or has_identifier_shape:
                    zip_candidate_rows.append({
                        "candidate": part,
                        "source_path": member_path,
                        "matched_terms": ",".join(matched_terms),
                        "evidence_status": "observed",
                        "interpretation": "ZIP path/name pattern only; not proof of an independent flight or session.",
                    })

            parts = list(Path(member_path).parts)
            sensors = {part.lower() for part in parts if part.lower() in {"rgb", "thermal", "visible", "infrared", "ir"}}
            if sensors and not record["is_directory"]:
                sensor = "rgb" if "rgb" in sensors or "visible" in sensors else "thermal"
                canonical_parts = [part.lower() for part in parts if part.lower() not in {"rgb", "thermal", "visible", "infrared", "ir"}]
                canonical = "/".join(canonical_parts[:-1]) if canonical_parts else "."
                existing = next((row for row in zip_sensor_rows if row["canonical_path_without_sensor"] == canonical), None)
                if existing is None:
                    existing = {
                        "canonical_path_without_sensor": canonical,
                        "rgb_filenames": set(),
                        "thermal_filenames": set(),
                    }
                    zip_sensor_rows.append(existing)
                existing[f"{sensor}_filenames"].add(record["filename"].lower())

    for row in zip_sensor_rows:
        rgb_names = row.pop("rgb_filenames")
        thermal_names = row.pop("thermal_filenames")
        row.update({
            "rgb_file_count": len(rgb_names),
            "thermal_file_count": len(thermal_names),
            "shared_filename_count": len(rgb_names & thermal_names),
            "evidence_status": "observed" if rgb_names and thermal_names else "unknown",
            "interpretation": (
                "RGB and thermal occur under a shared canonical ZIP path; verify timestamps/metadata before grouping."
                if rgb_names and thermal_names else
                "No paired ZIP path observed; this does not prove different flights or sessions."
            ),
        })

print(f"ZIP metadata files matching investigation terms: {len(zip_metadata_matches):,}")
print(f"ZIP candidate grouping path/name observations: {len(zip_candidate_rows):,}")
print(f"ZIP RGB/thermal canonical groups: {len(zip_sensor_rows):,}")

ZIP metadata files matching investigation terms: 0
ZIP candidate grouping path/name observations: 0
ZIP RGB/thermal canonical groups: 0


In [23]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def write_csv(filename: str, rows: list[dict], fieldnames: list[str]) -> None:
    output_path = REPORT_DIR / filename
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

extension_rows = [
    {"extension": extension, "file_count": count}
    for extension, count in sorted(extension_counts.items(), key=lambda item: (-item[1], item[0]))
]
directory_extension_rows = [
    {"relative_directory": directory, "extension": extension, "file_count": count}
    for (directory, extension), count in sorted(extension_by_directory.items())
]
directory_rows = [
    {"relative_directory": directory, "file_count": count}
    for directory, count in sorted(directory_file_counts.items())
]
pattern_rows = [
    {"pattern_type": label, "match_count": pattern_counts[label], "examples": "; ".join(pattern_examples[label])}
    for label in sorted(pattern_counts)
]

write_csv("extension_counts.csv", extension_rows, ["extension", "file_count"])
write_csv("directory_extension_counts.csv", directory_extension_rows, ["relative_directory", "extension", "file_count"])
write_csv("directory_file_counts.csv", directory_rows, ["relative_directory", "file_count"])
write_csv("metadata_term_matches.csv", metadata_matches, ["relative_path", "extension", "filename_terms", "content_terms", "terms", "content_status", "evidence_status"])
write_csv("metadata_term_snippets.csv", metadata_snippets, ["relative_path", "line_number", "terms", "snippet"])
write_csv("naming_patterns.csv", pattern_rows, ["pattern_type", "match_count", "examples"])
write_csv("grouping_candidates.csv", candidate_rows[:500], ["candidate", "source_path", "matched_terms", "evidence_status", "interpretation"])
write_csv("rgb_thermal_grouping.csv", sensor_rows[:500], ["canonical_path_without_sensor", "rgb_file_count", "thermal_file_count", "shared_filename_count", "evidence_status", "interpretation"])

(REPORT_DIR / "directory_tree.txt").write_text("\n".join(tree_lines) + "\n", encoding="utf-8")

reported_documented = sum(row["evidence_status"] == "documented" for row in metadata_matches)
reported_observed = sum(row["evidence_status"] == "observed" for row in metadata_matches) + len(candidate_rows)
summary = f"""WiSARv1 organization investigation
=================================
Dataset root: {root_resolved}
Files observed: {len(file_records):,}
Directories observed: {len(directory_records):,}
Metadata-like files observed: {len(metadata_records):,}
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: {reported_documented:,} metadata files contain search terms
observed: {reported_observed:,} filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports
-------
{chr(10).join(sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()))}
"""
(REPORT_DIR / "organization_investigation_report.txt").write_text(summary, encoding="utf-8")

print(summary)


WiSARv1 organization investigation
Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Files observed: 1
Directories observed: 1
Metadata-like files observed: 0
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: 0 metadata files contain search terms
observed: 0 filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports


In [24]:
zip_report_dir = REPORT_DIR
zip_report_dir.mkdir(parents=True, exist_ok=True)

if zip_status == "observed":
    zip_extension_rows = [
        {"extension": extension, "file_count": count}
        for extension, count in sorted(zip_extension_counts.items(), key=lambda item: (-item[1], item[0]))
    ]
    zip_pattern_rows = [
        {"pattern_type": label, "match_count": zip_pattern_counts[label], "examples": "; ".join(zip_pattern_examples[label])}
        for label in sorted(zip_pattern_counts)
    ]
    write_csv("zip_extension_counts.csv", zip_extension_rows, ["extension", "file_count"])
    write_csv("zip_metadata_term_matches.csv", zip_metadata_matches, ["member_path", "extension", "filename_terms", "content_terms", "terms", "content_status", "evidence_status"])
    write_csv("zip_metadata_term_snippets.csv", zip_metadata_snippets, ["member_path", "line_number", "terms", "snippet"])
    write_csv("zip_naming_patterns.csv", zip_pattern_rows, ["pattern_type", "match_count", "examples"])
    write_csv("zip_grouping_candidates.csv", zip_candidate_rows[:500], ["candidate", "source_path", "matched_terms", "evidence_status", "interpretation"])
    write_csv("zip_rgb_thermal_grouping.csv", zip_sensor_rows[:500], ["canonical_path_without_sensor", "rgb_file_count", "thermal_file_count", "shared_filename_count", "evidence_status", "interpretation"])
    (zip_report_dir / "zip_directory_tree.txt").write_text("\n".join(zip_tree_lines) + "\n", encoding="utf-8")

    zip_documented = sum(row["evidence_status"] == "documented" for row in zip_metadata_matches)
    zip_observed = sum(row["evidence_status"] == "observed" for row in zip_metadata_matches) + len(zip_candidate_rows)
    zip_summary = f"""WiSARDv1 ZIP organization investigation
=======================================
ZIP path: {zip_path}
ZIP members observed: {len(zip_member_records):,}
Image pixels opened: no
ZIP extracted/copied/moved/modified: no
Train/validation/test split created: no

Evidence labels
---------------
documented: {zip_documented:,} ZIP metadata/path records with metadata text terms
observed: {zip_observed:,} ZIP filename/content/path observations
unknown: different ZIP folder names are not treated as different flights without documentation or metadata

Metadata reads
--------------
Only non-image metadata-like members at or below MAX_METADATA_BYTES were read with ZipFile.open().
No image member was opened, decoded, resized, or extracted.

RGB/thermal
-----------
Shared canonical ZIP paths are labeled observed correspondence only.
Non-overlap is labeled unknown, not evidence of different flights or sessions.
"""
    (zip_report_dir / "zip_organization_investigation_report.txt").write_text(zip_summary, encoding="utf-8")
    print(zip_summary)
else:
    print("No WiSARDv1.zip detected under DATASET_ROOT; ZIP reports were not written.")


No WiSARDv1.zip detected under DATASET_ROOT; ZIP reports were not written.
